In [1]:
"""
==============================================================================
 CELL 1 — IMPORTS, REPRODUCIBILITY, DEVICE, & CONFIGURATION
 HPLC-GLUCOSE Fermentation — Transformer-DDPM Augmented CNN Pipeline
 Reused Architecture:
   - Transformer-DDPM (PyTorch): sinusoidal time embedding, EMA, cosine
     noise schedule, DDIM sampler, grid search over (nhead, num_layers,
     dim_feedforward)
   - Residual 1D-CNN (TensorFlow): residual blocks with LayerNorm + LeakyReLU,
     GlobalAveragePooling, cosine LR, EarlyStopping, ModelCheckpoint
   - Biological filtering: 5% / 2% / 1% tolerance bands around real trajectories
   - Validation suite: PCA, t-SNE, correlation heatmap, KS test,
     Wasserstein distance, Jensen-Shannon divergence
   - 7-metric CNN evaluation (R2, MAE, RMSE, Pearson r, Precision, Recall, F1)
==============================================================================
"""
import os, math, copy, time, random, warnings, itertools
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance, entropy
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              precision_score, recall_score, f1_score)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset

import tensorflow as tf
from tensorflow.keras import layers, regularizers, callbacks

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

# ── Reproducibility & Device ─────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); tf.random.set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device : {DEVICE}")
print(f"TensorFlow version: {tf.__version__}")

# ── File Paths & Directories ─────────────────────────────────────────────────
DATA_PATH = "/kaggle/input/datasets/bedashrutimajumdar/hplc-glucose/HPLC-GLUCOSE.xlsx"
BASE      = "/kaggle/working/HPLC-DDPM-CNN"
FIG_DIR   = os.path.join(BASE, "figures")
SYN_DIR   = os.path.join(BASE, "synthetic_data")
MET_DIR   = os.path.join(BASE, "metrics")
MDL_DIR   = os.path.join(BASE, "models")

for d in [FIG_DIR, SYN_DIR, MET_DIR, MDL_DIR]:
    os.makedirs(d, exist_ok=True)

SAVEFIG_KW = dict(dpi=300, bbox_inches='tight')
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# ── Global Hyperparameters & Dataset Constants ──────────────────────────────
TIME_COL      = 'Hour'
REPLICATES    = ['Replica-1', 'Replica-2', 'Replica-3']
SPECIES       = ['Glucose', 'Xylose', 'Ethanol', 'Glycerol', 'Acetate']
INPUT_COLS    = [TIME_COL, 'OD'] + [f"{s}_smooth" for s in SPECIES]

SG_WIN        = 5
SG_POLY       = 2
WINDOW        = 3

L2_LAMBDA     = 1e-3
MAX_EPOCHS    = 500
BATCH_SIZE    = 16
PATIENCE      = 60
LR_START      = 1e-3
LR_END        = 1e-6

T_DIFF        = 1000
N_SYNTH       = 8000
BATCH_GEN     = 500
FILTER_FRACTIONS = {'5pct': 0.05, '2pct': 0.02, '1pct': 0.01}

print(f"Output base directory: {BASE}")

PyTorch device : cpu
TensorFlow version: 2.20.0
Output base directory: /kaggle/working/HPLC-DDPM-CNN


In [2]:
"""
==============================================================================
 CELL 2 — LOAD & INSPECT EXPERIMENTAL REPLICATES
==============================================================================
"""
import os
import matplotlib.pyplot as plt
import pandas as pd

xl = pd.ExcelFile(DATA_PATH)
print("Sheet names in workbook:", xl.sheet_names)

# Load the three experimental replicate sheets
raw_replicates = {}
for rep_name in REPLICATES:
    df = pd.read_excel(DATA_PATH, sheet_name=rep_name)
    df.columns = df.columns.str.strip()
    
    print(f"\n--- Inspecting {rep_name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    
    # Ensure all target columns are numeric and sorted chronologically
    target_cols = [TIME_COL, 'OD'] + SPECIES
    for col in target_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    raw_replicates[rep_name] = df
    print("Missing values per column:\n", df[target_cols].isna().sum())
    print("Head:\n", df.head(3))

# Assign shorthand handles for convenient reference downstream
rep1, rep2, rep3 = [raw_replicates[r] for r in REPLICATES]

# ── Plot Raw Experimental Trajectories ──────────────────────────────────────

# ── 1. Set Publication Typography & Figure Sizing ───────────────────────────
# Note: Canvas is compact (10x6.5 in) so text remains large when inserted into Word/LaTeX
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

plot_targets = ['OD'] + SPECIES
colors = ['#1F77B4', '#FF7F0E', '#2CA02C']
markers = ['o', 's', '^']

# ── 2. Initialize Grid Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5), sharex=True, dpi=300)
axes = axes.flatten()

for i, col in enumerate(plot_targets):
    ax = axes[i]
    
    # Plot each replicate trajectory
    for r_idx, rep_name in enumerate(REPLICATES):
        d = raw_replicates[rep_name]
        ax.plot(
            d[TIME_COL], d[col],
            marker=markers[r_idx],
            markersize=6,
            linewidth=2.0,
            color=colors[r_idx],
            label=rep_name,
            alpha=0.9
        )
    
    unit = "OD₆₀₀" if col == 'OD' else "g/L"
    ax.set_title(col, fontweight='bold', pad=8)
    
    # Axis Labels
    if i >= 3:
        ax.set_xlabel("Time (h)", fontweight='bold', labelpad=4)
        
    y_label_text = f"Absorbance ({unit})" if col == 'OD' else f"Conc. ({unit})"
    ax.set_ylabel(y_label_text, fontweight='bold', labelpad=4)
    
    # Spines, Ticks, & Grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', color='#cccccc')
    ax.tick_params(direction='out', length=5, width=1.2)

# ── 3. Shared Legend & Layout Refinement ────────────────────────────────────
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=3,
    frameon=False,
    handletextpad=0.5,
    columnspacing=2.5
)

plt.tight_layout(rect=[0, 0.05, 1, 1.0])

# ── 4. Save High-Resolution Outputs ──────────────────────────────────────────
save_path_png = os.path.join(FIG_DIR, "Fig1_Raw_Trajectories.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig1_Raw_Trajectories.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig1_Raw_Trajectories.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)
print(f"Publication figure re-exported successfully:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")

Sheet names in workbook: ['Replica-1', 'Replica-2', 'Replica-3', 'Sheet5', 'Standard (FluxTKLTef1)', 'Standard (FluxTKLtef-3) ', 'Standard (Flux-TKLtef5']

--- Inspecting Replica-1 ---
Shape: (17, 7)
Columns: ['Hour', 'OD', 'Glucose', 'Xylose', 'Ethanol', 'Glycerol', 'Acetate']
Missing values per column:
 Hour        0
OD          0
Glucose     0
Xylose      0
Ethanol     0
Glycerol    0
Acetate     0
dtype: int64
Head:
    Hour    OD    Glucose     Xylose   Ethanol  Glycerol  Acetate
0   0.0  0.05  20.403862  10.253298  0.000000  0.002101      0.0
1   8.0  0.66  17.478815  10.017993  0.500581  0.042476      0.0
2  11.0  2.65  15.456105   9.835553  1.100842  0.088733      0.0

--- Inspecting Replica-2 ---
Shape: (17, 7)
Columns: ['Hour', 'OD', 'Glucose', 'Xylose', 'Ethanol', 'Glycerol', 'Acetate']
Missing values per column:
 Hour        0
OD          0
Glucose     0
Xylose      0
Ethanol     0
Glycerol    0
Acetate     0
dtype: int64
Head:
    Hour    OD    Glucose     Xylose   Ethanol

In [3]:
"""
==============================================================================
 CELL 3 — PREPROCESSING: MISSING VALUE HANDLING, SAVITZKY-GOLAY SMOOTHING,
          & SMOOTHED EXCEL EXPORT
==============================================================================
"""
def clean_and_smooth_replicate(df_rep: pd.DataFrame, target_cols) -> pd.DataFrame:
    df_rep = df_rep.copy().reset_index(drop=True)
    
    # Missing-value handling: linear interpolation -> ffill -> bfill
    for col in [TIME_COL] + target_cols:
        df_rep[col] = df_rep[col].interpolate(method='linear').ffill().bfill()
        
    # Savitzky-Golay polynomial smoothing
    for col in target_cols:
        vals = df_rep[col].values.astype(float)
        if np.allclose(vals, 0.0):
            df_rep[f"{col}_smooth"] = vals.copy()
        else:
            df_rep[f"{col}_smooth"] = savgol_filter(vals, SG_WIN, SG_POLY)
            
    return df_rep

# Process each replicate sheet
smooth_targets = ['OD'] + SPECIES
processed_replicates = {}

for rep_name in REPLICATES:
    sub = raw_replicates[rep_name].copy()
    sub_processed = clean_and_smooth_replicate(sub, smooth_targets)
    processed_replicates[rep_name] = sub_processed

# Re-assign convenience handles
rep1, rep2, rep3 = [processed_replicates[r] for r in REPLICATES]

# ── Export Smoothed Data to Excel File ───────────────────────────────────────
smoothed_excel_path = os.path.join(MET_DIR, "HPLC_GLUCOSE_Smoothed.xlsx")
with pd.ExcelWriter(smoothed_excel_path) as writer:
    for rep_name, df_smoothed in processed_replicates.items():
        df_smoothed.to_excel(writer, sheet_name=rep_name, index=False)

print(f"Smoothed dataset successfully exported to:\n  {smoothed_excel_path}")

# ── Plot Raw vs. Smoothed Trajectories ──────────────────────────────────────

# ── 1. Set Publication Typography & Figure Sizing (From Cell 2) ──────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

colors = ['#1F77B4', '#FF7F0E', '#2CA02C']
markers = ['o', 's', '^']

# ── 2. Initialize Grid Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5), sharex=True, dpi=300)
axes = axes.flatten()

for i, col in enumerate(smooth_targets):
    ax = axes[i]
    
    for r_idx, rep_name in enumerate(REPLICATES):
        d = processed_replicates[rep_name]
        # Raw data scatter points
        ax.plot(
            d[TIME_COL], d[col],
            marker=markers[r_idx],
            linestyle='None',
            markersize=5,
            color=colors[r_idx],
            alpha=0.45,
            label=f"{rep_name} (Raw)"
        )
        # Smoothed trajectory line
        ax.plot(
            d[TIME_COL], d[f"{col}_smooth"],
            linestyle='-',
            linewidth=2.0,
            color=colors[r_idx],
            alpha=0.95,
            label=f"{rep_name} (Smoothed)"
        )
        
    unit = "OD₆₀₀" if col == 'OD' else "g/L"
    ax.set_title(col, fontweight='bold', pad=8)
    
    # Axis Labels
    if i >= 3:
        ax.set_xlabel("Time (h)", fontweight='bold', labelpad=4)
        
    y_label_text = f"Absorbance ({unit})" if col == 'OD' else f"Conc. ({unit})"
    ax.set_ylabel(y_label_text, fontweight='bold', labelpad=4)
    
    # Spines, Ticks, & Grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', color='#cccccc')
    ax.tick_params(direction='out', length=5, width=1.2)

# ── 3. Shared Legend & Layout Refinement ────────────────────────────────────
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=3,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.5,
    fontsize=11
)

plt.tight_layout(rect=[0, 0.06, 1, 1.0])

# ── 4. Save High-Resolution Outputs ──────────────────────────────────────────
save_path_png = os.path.join(FIG_DIR, "Fig2_Smoothing_Trajectories.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig2_Smoothing_Trajectories.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig2_Smoothing_Trajectories.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)
print(f"Publication smoothing figure re-exported successfully:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")

Smoothed dataset successfully exported to:
  /kaggle/working/HPLC-DDPM-CNN/metrics/HPLC_GLUCOSE_Smoothed.xlsx
Publication smoothing figure re-exported successfully:
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig2_Smoothing_Trajectories.png
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig2_Smoothing_Trajectories.pdf
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig2_Smoothing_Trajectories.svg


In [4]:
"""
==============================================================================
 CELL 4 — CENTRAL FINITE DIFFERENCE (CFD) KINETIC RATES & EXCEL EXPORT
==============================================================================
"""
def cfd_rate(y_smooth: np.ndarray, t: np.ndarray) -> np.ndarray:
    """
    Central finite difference derivative calculation:
    - Central difference for interior points: (y[i+1] - y[i-1]) / (t[i+1] - t[i-1])
    - Forward difference at start edge:       (y[1] - y[0]) / (t[1] - t[0])
    - Backward difference at end edge:        (y[-1] - y[-2]) / (t[-1] - t[-2])
    """
    dydt = np.zeros(len(t))
    if len(t) < 2:
        return dydt
    dydt[1:-1] = (y_smooth[2:] - y_smooth[:-2]) / (t[2:] - t[:-2])
    dydt[0]    = (y_smooth[1]  - y_smooth[0])   / (t[1]  - t[0])
    dydt[-1]   = (y_smooth[-1] - y_smooth[-2])  / (t[-1] - t[-2])
    return dydt

# Compute kinetic rates across all replicates
rate_targets = ['OD'] + SPECIES

for rep_name in REPLICATES:
    d = processed_replicates[rep_name]
    t = d[TIME_COL].values.astype(float)
    
    for col in rate_targets:
        d[f"{col}_rate"] = cfd_rate(d[f"{col}_smooth"].values, t)

# Re-assign convenience handles
rep1, rep2, rep3 = [processed_replicates[r] for r in REPLICATES]

# ── Export Dataset with Rates to Excel File ─────────────────────────────────
rates_excel_path = os.path.join(MET_DIR, "HPLC_GLUCOSE_Rates.xlsx")
with pd.ExcelWriter(rates_excel_path) as writer:
    for rep_name, df_data in processed_replicates.items():
        df_data.to_excel(writer, sheet_name=rep_name, index=False)

print(f"Rates dataset successfully exported to:\n  {rates_excel_path}")

# ── Plot CFD Kinetic Rates ──────────────────────────────────────────────────

# ── 1. Set Publication Typography & Figure Sizing (From Cell 2) ──────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

colors = ['#1F77B4', '#FF7F0E', '#2CA02C']
markers = ['o', 's', '^']

# ── 2. Initialize Grid Figure ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(10, 6.5), sharex=True, dpi=300)
axes = axes.flatten()

for i, col in enumerate(rate_targets):
    ax = axes[i]
    
    for r_idx, rep_name in enumerate(REPLICATES):
        d = processed_replicates[rep_name]
        ax.plot(
            d[TIME_COL], d[f"{col}_rate"],
            marker=markers[r_idx],
            markersize=6,
            linewidth=2.0,
            color=colors[r_idx],
            label=rep_name,
            alpha=0.9
        )
        
    ax.axhline(0, color='#666666', linewidth=1.0, linestyle=':')
    unit = "OD₆₀₀/h" if col == 'OD' else "g/L/h"
    ax.set_title(col, fontweight='bold', pad=8)
    
    # Axis Labels
    if i >= 3:
        ax.set_xlabel("Time (h)", fontweight='bold', labelpad=4)
        
    ax.set_ylabel(f"Rate ({unit})", fontweight='bold', labelpad=4)
    
    # Spines, Ticks, & Grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', color='#cccccc')
    ax.tick_params(direction='out', length=5, width=1.2)

# ── 3. Shared Legend & Layout Refinement ────────────────────────────────────
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=3,
    frameon=False,
    handletextpad=0.5,
    columnspacing=2.5
)

plt.tight_layout(rect=[0, 0.05, 1, 1.0])

# ── 4. Save High-Resolution Outputs ──────────────────────────────────────────
save_path_png = os.path.join(FIG_DIR, "Fig3_CFDRates_Trajectories.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig3_CFDRates_Trajectories.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig3_CFDRates_Trajectories.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)
print(f"Publication kinetic-rate figure re-exported successfully:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")

Rates dataset successfully exported to:
  /kaggle/working/HPLC-DDPM-CNN/metrics/HPLC_GLUCOSE_Rates.xlsx
Publication kinetic-rate figure re-exported successfully:
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig3_CFDRates_Trajectories.png
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig3_CFDRates_Trajectories.pdf
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig3_CFDRates_Trajectories.svg


In [5]:
"""
==============================================================================
 CELL 5 — NORMALIZATION, TARGET SCALING, & SLIDING WINDOW CONSTRUCTION
 Strict Replicate Partitioning: Training (Replica-1 + Replica-2) vs Held-out (Replica-3)
==============================================================================
"""
def make_windows(X_sc: np.ndarray, y: np.ndarray, window: int):
    """
    Constructs 3D sequence windows from 2D normalized feature matrices.
    Returns:
        Xs: Array of shape (N_windows, window, N_features)
        ys: Target vector corresponding to the final timestep of each window
    """
    Xs, ys = [], []
    for k in range(len(X_sc) - window + 1):
        Xs.append(X_sc[k: k + window])
        ys.append(y[k + window - 1])
    if len(Xs) == 0:
        return (np.zeros((0, window, X_sc.shape[1]), dtype=np.float32),
                np.zeros((0,), dtype=np.float32))
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# Define feature columns for scaling and feature extraction
INPUT_COLS = [TIME_COL, 'OD'] + [f"{s}_smooth" for s in SPECIES]
N_IN = len(INPUT_COLS)

# ── Feature Scaler (MinMaxScaler) ────────────────────────────────────────────
# Fit strictly on Replica-1 and Replica-2 to prevent data leakage from Replica-3
train_feature_rows = pd.concat([rep1[INPUT_COLS], rep2[INPUT_COLS]], axis=0, ignore_index=True)
scaler_X = MinMaxScaler().fit(train_feature_rows.values)

# Transform feature matrices for all replicates
X1_sc = scaler_X.transform(rep1[INPUT_COLS].values)
X2_sc = scaler_X.transform(rep2[INPUT_COLS].values)
X3_sc = scaler_X.transform(rep3[INPUT_COLS].values)

print(f"Feature Scaler fitted on training replicates (Replica-1 + Replica-2).")
print(f"Input Features ({N_IN}): {INPUT_COLS}")
print(f"Window Size: {WINDOW} time-steps")

# Verify window shapes across replicates
X1w_dummy, _ = make_windows(X1_sc, rep1[f"{SPECIES[0]}_rate"].values, WINDOW)
X2w_dummy, _ = make_windows(X2_sc, rep2[f"{SPECIES[0]}_rate"].values, WINDOW)
X3w_dummy, _ = make_windows(X3_sc, rep3[f"{SPECIES[0]}_rate"].values, WINDOW)

print(f"\nConstructed Sliding Window Shapes:")
print(f"  Replica-1 Windows : {X1w_dummy.shape}")
print(f"  Replica-2 Windows : {X2w_dummy.shape}")
print(f"  Replica-3 Windows : {X3w_dummy.shape} (Held-out Test)")

Feature Scaler fitted on training replicates (Replica-1 + Replica-2).
Input Features (7): ['Hour', 'OD', 'Glucose_smooth', 'Xylose_smooth', 'Ethanol_smooth', 'Glycerol_smooth', 'Acetate_smooth']
Window Size: 3 time-steps

Constructed Sliding Window Shapes:
  Replica-1 Windows : (15, 3, 7)
  Replica-2 Windows : (15, 3, 7)
  Replica-3 Windows : (15, 3, 7) (Held-out Test)


In [6]:
"""
==============================================================================
 CELL 6 — TRANSFORMER-DDPM (PyTorch)
 Trains on FULL trajectories (not CNN windows) to generate whole synthetic
 growth trajectories, exactly mirroring the reference design.
 
 STRUCTURAL ADAPTATION NOTES (HPLC-GLUCOSE):
 All replicates have a uniform length (17 hours), so padding is not strictly
 needed, but the masking logic from the reference is preserved intact (mask
 will simply be all 1s). The DDPM will exclusively train on Replica-1 and
 Replica-2 to prevent data leakage from the held-out Replica-3.
==============================================================================
"""
def cosine_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    acp = f / f[0]
    betas = torch.clamp(1 - acp[1:] / acp[:-1], 1e-5, 0.999)
    return betas.float()

betas = cosine_schedule(T_DIFF).to(DEVICE)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, 0)
sqrt_acp = torch.sqrt(alphas_cumprod)
sqrt_1m_acp = torch.sqrt(1.0 - alphas_cumprod)

def q_sample(x0, t, noise):
    sa = sqrt_acp[t].view(-1, 1, 1)
    s1m = sqrt_1m_acp[t].view(-1, 1, 1)
    return sa * x0 + s1m * noise

class SinusoidalEmb(nn.Module):
    def __init__(self, d_model):
        super().__init__(); self.d = d_model
    def forward(self, t):
        half = self.d // 2
        freqs = torch.exp(-torch.arange(half, device=t.device).float()
                           * (math.log(10000) / max(half - 1, 1)))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([args.sin(), args.cos()], dim=-1)

class TransformerDenoiser(nn.Module):
    def __init__(self, seq_len, n_feat, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(n_feat, d_model)
        self.out_proj = nn.Linear(d_model, n_feat)
        self.time_emb = SinusoidalEmb(d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
    def forward(self, x, t):
        h = self.in_proj(x) + self.pos_enc
        t_emb = self.time_emb(t).unsqueeze(1)
        return self.out_proj(self.encoder(h + t_emb))

class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay; self.shadow = copy.deepcopy(model).eval()
    @torch.no_grad()
    def update(self, model):
        for s, p in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)
    def model(self):
        return self.shadow

@torch.no_grad()
def ddim_sample(model, n_samples, seq_len, n_feat, ddim_steps=200, eta=0.0):
    model.eval()
    step_seq = list(reversed(np.linspace(0, T_DIFF - 1, ddim_steps, dtype=int)))
    x = torch.randn(n_samples, seq_len, n_feat, device=DEVICE)
    for i, t_idx in enumerate(step_seq):
        tb = torch.full((n_samples,), t_idx, device=DEVICE, dtype=torch.long)
        pred_eps = model(x, tb)
        acp_t = alphas_cumprod[t_idx]
        acp_prev = (alphas_cumprod[step_seq[i + 1]] if i + 1 < len(step_seq)
                    else torch.tensor(1.0, device=DEVICE))
        x0_pred = ((x - sqrt_1m_acp[t_idx] * pred_eps) / sqrt_acp[t_idx]).clamp(-5.0, 5.0)
        sigma = eta * torch.sqrt((1 - acp_prev) / (1 - acp_t) * (1 - acp_t / acp_prev))
        x = torch.sqrt(acp_prev) * x0_pred + torch.sqrt(
            torch.clamp(1 - acp_prev - sigma ** 2, min=0.0)) * pred_eps
    return x.cpu().numpy()

def augment_sequences(X: np.ndarray, mask: np.ndarray, n_aug=300, noise_std=0.08):
    """Jitter + pairwise interpolation corpus growth."""
    N = len(X)
    rng = np.random.default_rng(SEED)
    jitter_idx = rng.integers(0, N, n_aug)
    jitter = X[jitter_idx] + rng.standard_normal((n_aug, X.shape[1], X.shape[2])) * noise_std
    jitter_mask = mask[jitter_idx]
    ia = rng.integers(0, N, n_aug); ib = rng.integers(0, N, n_aug)
    alpha = rng.uniform(0.2, 0.8, (n_aug, 1, 1)).astype(np.float32)
    interp = alpha * X[ia] + (1 - alpha) * X[ib]
    interp_mask = np.minimum(mask[ia], mask[ib])
    X_out = np.concatenate([X, jitter.astype(np.float32), interp.astype(np.float32)], axis=0)
    mask_out = np.concatenate([mask, jitter_mask, interp_mask], axis=0)
    return X_out, mask_out

def masked_mse(pred, target, mask):
    diff2 = (pred - target) ** 2 * mask
    denom = mask.sum().clamp(min=1.0)
    return diff2.sum() / denom

def train_ddpm_config(config: dict, X_t: torch.Tensor, mask_t: torch.Tensor,
                       epochs=800, batch=8, lr=3e-4):
    model = TransformerDenoiser(**config).to(DEVICE)
    ema_obj = EMA(model)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)
    batch = min(batch, len(X_t))
    loader = DataLoader(TensorDataset(X_t, mask_t), batch_size=batch,
                         shuffle=True, drop_last=(len(X_t) > batch))
    losses = []
    for epoch in range(1, epochs + 1):
        model.train(); ep_loss = 0.0; n_seen = 0
        for b, m in loader:
            b = b.to(DEVICE); m = m.to(DEVICE)
            t = torch.randint(0, T_DIFF, (len(b),), device=DEVICE)
            noise = torch.randn_like(b)
            xt = q_sample(b, t, noise)
            loss = masked_mse(model(xt, t), noise, m)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); ema_obj.update(model)
            ep_loss += loss.item() * len(b); n_seen += len(b)
        sched.step()
        losses.append(ep_loss / max(1, n_seen))
    return model, ema_obj, losses

def score_config(ema_obj, real_arr, mask_arr, seq_len, n_feat, scaler_conc):
    synth_logit = ddim_sample(ema_obj.model(), n_samples=min(100, max(20, real_arr.shape[0]*5)),
                               seq_len=seq_len, n_feat=n_feat, ddim_steps=100)
    synth_raw = scaler_conc.inverse_transform(
        sigmoid_np(synth_logit).reshape(-1, n_feat)).reshape(synth_logit.shape[0], seq_len, n_feat)
    real_flat = real_arr.reshape(-1, n_feat)
    synth_flat = synth_raw.reshape(-1, n_feat)
    mse = np.mean((real_flat.mean(0) - synth_flat.mean(0)) ** 2)
    var_real = np.var(real_flat, axis=0).mean() + 1e-8
    mse_score = max(0.0, 1.0 - mse / var_real)
    r_scores = []
    for fi in range(n_feat):
        r_ts = real_arr[:, :, fi].mean(0)
        s_ts = synth_raw[:, :, fi].mean(0)
        L = min(len(r_ts), len(s_ts))
        r, _ = pearsonr(r_ts[:L], s_ts[:L])
        r_scores.append(max(0.0, float(r) if not np.isnan(r) else 0.0))
    return 0.5 * mse_score + 0.5 * np.mean(r_scores)

def logit_np(x, eps=1e-6):
    x = np.clip(x, eps, 1 - eps)
    return np.log(x / (1 - x)).astype(np.float32)

def sigmoid_np(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)

print("Transformer-DDPM components defined.")

Transformer-DDPM components defined.


In [7]:
"""
==============================================================================
 CELL 7 — TRAIN DDPM: GRID SEARCH + FULL TRAINING + LOSS PLOT
==============================================================================
"""
SEARCH_SPACE = {"nhead": [2, 4], "num_layers": [2, 3], "dim_feedforward": [128, 256]}
D_MODEL       = 64
SEARCH_EPOCHS = 150
FULL_EPOCHS   = 1500

# Joint DDPM targets: biomass proxy (OD) + all 5 chemical species
DDPM_TARGETS = ['OD'] + SPECIES
n_feat = len(DDPM_TARGETS)
train_replicates = ['Replica-1', 'Replica-2']

seq_len = max(len(processed_replicates[r]) for r in train_replicates)
real_arr = np.zeros((len(train_replicates), seq_len, n_feat), dtype=np.float32)
mask_arr = np.ones((len(train_replicates), seq_len, n_feat), dtype=np.float32)

for r_idx, r_name in enumerate(train_replicates):
    d = processed_replicates[r_name]
    for fi, col in enumerate(DDPM_TARGETS):
        real_arr[r_idx, :, fi] = d[col].values

flat_real = real_arr.reshape(-1, n_feat)
scaler_conc = MinMaxScaler(feature_range=(0.02, 0.98)).fit(flat_real)
norm_arr = scaler_conc.transform(flat_real).reshape(len(train_replicates), seq_len, n_feat)
logit_arr = logit_np(norm_arr)

# Corpus expansion via jitter and pairwise interpolation
X_aug, mask_aug = augment_sequences(logit_arr, mask_arr, n_aug=300)
X_train_t = torch.from_numpy(X_aug)
mask_train_t = torch.from_numpy(mask_aug)

print(f"\n[HPLC-GLUCOSE] DDPM training corpus shape: {X_train_t.shape} "
      f"(seq_len={seq_len}, n_feat={n_feat})")

# ── 1. Grid Search over Transformer Denoiser Configs ─────────────────────────
print("[HPLC-GLUCOSE] Grid search over Transformer configs ...")
configs = list(itertools.product(SEARCH_SPACE['nhead'], SEARCH_SPACE['num_layers'],
                                  SEARCH_SPACE['dim_feedforward']))
search_results = []

for nhead, num_layers, dff in configs:
    cfgd = dict(d_model=D_MODEL, nhead=nhead, num_layers=num_layers,
                dim_feedforward=dff, seq_len=seq_len, n_feat=n_feat)
    _, e, lh = train_ddpm_config(cfgd, X_train_t, mask_train_t, epochs=SEARCH_EPOCHS)
    score = score_config(e, real_arr, mask_arr, seq_len, n_feat, scaler_conc)
    print(f"  nhead={nhead} L={num_layers} dff={dff:3d} -> "
          f"score={score:.4f}  final_loss={lh[-1]:.5f}")
    search_results.append({'config': cfgd, 'score': score})

search_results.sort(key=lambda x: -x['score'])
best_cfg = search_results[0]['config']
print(f"[HPLC-GLUCOSE] Best config: {best_cfg} | Score: {search_results[0]['score']:.4f}")

# ── 2. Full DDPM Training on Optimal Architecture ───────────────────────────
print(f"[HPLC-GLUCOSE] Full training ({FULL_EPOCHS} epochs) on best config ...")
best_model, best_ema, best_losses = train_ddpm_config(
    best_cfg, X_train_t, mask_train_t, epochs=FULL_EPOCHS)

# ── 3. Save Publication High-Resolution Training Loss Plot ──────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
ax.plot(best_losses, color='#1D3557', linewidth=2.0, label='Masked MSE Loss')

ax.set_title("DDPM Training Convergence", fontweight='bold', pad=10)
ax.set_xlabel("Epoch", fontweight='bold', labelpad=6)
ax.set_ylabel("Masked MSE Loss", fontweight='bold', labelpad=6)

# Spines, Grid, & Ticks Styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, linestyle='--', color='#cccccc')
ax.tick_params(direction='out', length=5, width=1.2)

plt.tight_layout()

# Save High-Resolution Outputs in Multiple Formats
save_path_png = os.path.join(FIG_DIR, "Fig3_DDPMTraining_HPLC.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig3_DDPMTraining_HPLC.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig3_DDPMTraining_HPLC.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)

# Save DDPM state dictionary for downstream sampling and filtering cells
ddpm_state = dict(
    best_ema=best_ema, best_cfg=best_cfg, scaler_conc=scaler_conc,
    seq_len=seq_len, n_feat=n_feat, real_arr=real_arr, mask_arr=mask_arr,
    ddpm_targets=DDPM_TARGETS
)

print(f"\nDDPM training complete. Saved publication loss plots:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")


[HPLC-GLUCOSE] DDPM training corpus shape: torch.Size([602, 17, 6]) (seq_len=17, n_feat=6)
[HPLC-GLUCOSE] Grid search over Transformer configs ...
  nhead=2 L=2 dff=128 -> score=0.9994  final_loss=0.12396
  nhead=2 L=2 dff=256 -> score=0.9995  final_loss=0.11516
  nhead=2 L=3 dff=128 -> score=0.9998  final_loss=0.09666
  nhead=2 L=3 dff=256 -> score=0.9999  final_loss=0.10478
  nhead=4 L=2 dff=128 -> score=0.9997  final_loss=0.12986
  nhead=4 L=2 dff=256 -> score=0.9998  final_loss=0.10406
  nhead=4 L=3 dff=128 -> score=0.9999  final_loss=0.10442
  nhead=4 L=3 dff=256 -> score=0.9999  final_loss=0.10323
[HPLC-GLUCOSE] Best config: {'d_model': 64, 'nhead': 4, 'num_layers': 3, 'dim_feedforward': 256, 'seq_len': 17, 'n_feat': 6} | Score: 0.9999
[HPLC-GLUCOSE] Full training (1500 epochs) on best config ...

DDPM training complete. Saved publication loss plots:
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig3_DDPMTraining_HPLC.png
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig3_DDPMTraining_HPL

In [8]:
"""
==============================================================================
 CELL 8 — GENERATE SYNTHETIC TRAJECTORIES + SAVE CANDIDATES + OVERLAY PLOT
==============================================================================
"""
state = ddpm_state
seq_len, n_feat = state['seq_len'], state['n_feat']
scaler_conc     = state['scaler_conc']
ema_obj         = state['best_ema']
ddpm_targets    = state['ddpm_targets']

# ── 1. DDIM Sampling of Candidate Trajectories ──────────────────────────────
print(f"[HPLC-GLUCOSE] Generating {N_SYNTH} synthetic candidate trajectories...")
parts = []
n_batches = max(1, N_SYNTH // BATCH_GEN)

for b in range(n_batches):
    part = ddim_sample(ema_obj.model(), n_samples=BATCH_GEN, seq_len=seq_len,
                        n_feat=n_feat, ddim_steps=200, eta=0.0)
    parts.append(part)

synth_logit = np.concatenate(parts, axis=0)[:N_SYNTH]
synth_raw   = scaler_conc.inverse_transform(
    sigmoid_np(synth_logit).reshape(-1, n_feat)).reshape(-1, seq_len, n_feat)
synth_raw   = np.clip(synth_raw, 0.0, None)

print(f"[HPLC-GLUCOSE] Candidate pool array shape: {synth_raw.shape}")

# Exact time grid from experimental data (17 time-steps)
time_vec = rep1[TIME_COL].values.astype(float)

# Fast 3D Array setup for candidates
n_candidates = synth_raw.shape[0]

# ── 2. Vectorized SG Smoothing & CFD Kinetic Computation ────────────────────
print("[HPLC-GLUCOSE] Computing SG smoothing and CFD kinetic rates for candidates...")

# Vectorized Savitzky-Golay Filter across axis 1 (Time axis)
synth_smooth = savgol_filter(synth_raw, window_length=SG_WIN, polyorder=SG_POLY, axis=1)

# Vectorized Central Finite Difference (CFD) Rate Calculation
synth_rates = np.zeros_like(synth_smooth)
if seq_len >= 2:
    # Central difference for interior timesteps
    dt_interior = time_vec[2:] - time_vec[:-2]  # Shape: (seq_len - 2,)
    synth_rates[:, 1:-1, :] = (synth_smooth[:, 2:, :] - synth_smooth[:, :-2, :]) / dt_interior[None, :, None]
    
    # Forward difference at initial boundary
    dt_start = time_vec[1] - time_vec[0]
    synth_rates[:, 0, :] = (synth_smooth[:, 1, :] - synth_smooth[:, 0, :]) / dt_start
    
    # Backward difference at final boundary
    dt_end = time_vec[-1] - time_vec[-2]
    synth_rates[:, -1, :] = (synth_smooth[:, -1, :] - synth_smooth[:, -2, :]) / dt_end

# Fast tabular DataFrame construction
sid_vec = np.repeat(np.arange(n_candidates), seq_len)
time_tiled = np.tile(time_vec, n_candidates)

df_dict = {'synthetic_id': sid_vec, TIME_COL: time_tiled}

for fi, sp in enumerate(ddpm_targets):
    df_dict[sp] = synth_raw[:, :, fi].reshape(-1)
    df_dict[f"{sp}_smooth"] = synth_smooth[:, :, fi].reshape(-1)
    df_dict[f"{sp}_rate"]   = synth_rates[:, :, fi].reshape(-1)

cand_df = pd.DataFrame(df_dict)

# Save candidate trajectories
cand_out_path = os.path.join(SYN_DIR, "synthetic_candidates_HPLC.csv")
cand_df.to_csv(cand_out_path, index=False)
synthetic_candidates = cand_df

print(f"[HPLC-GLUCOSE] Saved candidate pool -> {cand_out_path} "
      f"({len(cand_df)} rows, {cand_df['synthetic_id'].nunique()} trajectories)")

# ── 3. Real vs. Synthetic Overlay Visualization (Publication Style) ──────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 14,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.6,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

colors  = ['#1F77B4', '#FF7F0E', '#2CA02C']
markers = ['o', 's', '^']

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5), sharex=True, dpi=300)
axes = axes.flatten()

rng_val = np.random.default_rng(SEED)
idx200  = rng_val.choice(synth_raw.shape[0], size=min(200, synth_raw.shape[0]), replace=False)

for i, sp in enumerate(ddpm_targets):
    ax = axes[i]
    fi = ddpm_targets.index(sp)
    syn_sub = synth_raw[idx200, :, fi]
    s_mean, s_std = syn_sub.mean(0), syn_sub.std(0)
    
    # Plot experimental replicate trajectories
    for r_idx, rep_name in enumerate(REPLICATES):
        d = processed_replicates[rep_name]
        ax.plot(
            d[TIME_COL], d[sp].values,
            marker=markers[r_idx],
            markersize=5,
            linewidth=1.8,
            color=colors[r_idx],
            label=rep_name,
            alpha=0.85
        )
        
    # Plot synthetic mean + 1 std band
    ax.plot(time_vec, s_mean, color='#534AB7', linewidth=2.2, linestyle='--', label='Synthetic (mean)')
    ax.fill_between(time_vec, s_mean - s_std, s_mean + s_std, color='#534AB7', alpha=0.18)
    
    unit = "OD₆₀₀" if sp == 'OD' else "g/L"
    ax.set_title(sp, fontweight='bold', pad=8)
    
    # Axis Labels
    if i >= 3:
        ax.set_xlabel("Time (h)", fontweight='bold', labelpad=4)
        
    y_label_text = f"Absorbance ({unit})" if sp == 'OD' else f"Conc. ({unit})"
    ax.set_ylabel(y_label_text, fontweight='bold', labelpad=4)
    
    # Spines, Ticks, & Grid
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', color='#cccccc')
    ax.tick_params(direction='out', length=5, width=1.2)

# ── 4. Shared Legend & Layout Refinement ────────────────────────────────────
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.04),
    ncol=4,
    frameon=False,
    handletextpad=0.4,
    columnspacing=1.8,
    fontsize=12
)

plt.tight_layout(rect=[0, 0.05, 1, 1.0])

# ── 5. Save High-Resolution Outputs ──────────────────────────────────────────
save_path_png = os.path.join(FIG_DIR, "Fig_RealVsSynthetic_HPLC.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig_RealVsSynthetic_HPLC.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig_RealVsSynthetic_HPLC.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)
print(f"Publication overlay figure re-exported successfully:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")

[HPLC-GLUCOSE] Generating 8000 synthetic candidate trajectories...
[HPLC-GLUCOSE] Candidate pool array shape: (8000, 17, 6)
[HPLC-GLUCOSE] Computing SG smoothing and CFD kinetic rates for candidates...
[HPLC-GLUCOSE] Saved candidate pool -> /kaggle/working/HPLC-DDPM-CNN/synthetic_data/synthetic_candidates_HPLC.csv (136000 rows, 8000 trajectories)
Publication overlay figure re-exported successfully:
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_RealVsSynthetic_HPLC.png
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_RealVsSynthetic_HPLC.pdf
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_RealVsSynthetic_HPLC.svg


In [9]:
"""
==============================================================================
 CELL 9 — BIOLOGICAL FILTERING: 5% / 2% / 1% TOLERANCE BANDS (FIXED)
 Uses Feature-Averaged MAE Filtering to prevent 0% acceptance at tight bands.
==============================================================================
"""
state        = ddpm_state
seq_len, n_feat = state['seq_len'], state['n_feat']
scaler_conc  = state['scaler_conc']
real_arr     = state['real_arr']      # Shape: (n_train_reps, 17, 6)
cand_df      = synthetic_candidates
ddpm_targets = state['ddpm_targets']

n_synth_traj = cand_df['synthetic_id'].nunique()

# ── 1. Scale Real Data & Build Mean Reference Trajectory ──────────────────────
real_flat = real_arr.reshape(-1, n_feat)
real_norm = scaler_conc.transform(real_flat).reshape(real_arr.shape)
mean_real_norm = real_norm.mean(axis=0)  # Shape: (17, 6)

# ── 2. Scale Synthetic Candidate Data ─────────────────────────────────────────
synth_conc_3d = np.zeros((n_synth_traj, seq_len, n_feat), dtype=np.float32)

for fi, sp in enumerate(ddpm_targets):
    for sid, grp in cand_df.groupby('synthetic_id'):
        grp = grp.sort_values(TIME_COL)
        synth_conc_3d[sid, :, fi] = grp[sp].values[:seq_len]

synth_norm_3d = scaler_conc.transform(
    synth_conc_3d.reshape(-1, n_feat)).reshape(n_synth_traj, seq_len, n_feat)

# ── 3. Biological Tolerance Band Filtering Sweep ──────────────────────────────
filtered_subsets = {}
log_lines = ["[HPLC-GLUCOSE] Biological tolerance-band filtering summary:"]

# Calculate Mean Absolute Error (MAE) per trajectory across time and channels
mae_per_traj = np.abs(synth_norm_3d - mean_real_norm[None, :, :]).mean(axis=(1, 2))

for tag, frac in FILTER_FRACTIONS.items():
    # Primary check: Trajectory MAE within tolerance
    accept_mask = mae_per_traj <= frac  # Shape: (n_synth_traj,)
    
    # Fallback safeguard: If strict threshold yields 0, select top 1% closest candidates
    if accept_mask.sum() == 0:
        top_k = max(10, int(0.01 * n_synth_traj))
        top_indices = np.argsort(mae_per_traj)[:top_k]
        accept_mask = np.zeros(n_synth_traj, dtype=bool)
        accept_mask[top_indices] = True
        log_lines.append(f"  tolerance={tag:>4s}: [Fallback Top-K] accepted {top_k:>4d}/{n_synth_traj} "
                         f"({(100.0 * top_k) / n_synth_traj:.2f}%)")
    else:
        n_accept = int(accept_mask.sum())
        pct_accept = (100.0 * n_accept) / n_synth_traj
        log_lines.append(f"  tolerance={tag:>4s}: accepted {n_accept:>4d}/{n_synth_traj} "
                         f"({pct_accept:.2f}%)")
    
    keep_ids  = np.where(accept_mask)[0]
    subset_df = cand_df[cand_df['synthetic_id'].isin(keep_ids)].copy()
    
    # Export filtered synthetic subset CSV
    out_path = os.path.join(SYN_DIR, f"synthetic_subset_{tag}.csv")
    subset_df.to_csv(out_path, index=False)
    filtered_subsets[tag] = subset_df

# Print and log filtering results
print("\n".join(log_lines))

filter_log_path = os.path.join(MET_DIR, "filtering_log_HPLC.txt")
with open(filter_log_path, "w") as f:
    f.write("\n".join(log_lines))

print(f"\nFiltering complete. Log saved -> {filter_log_path}")

[HPLC-GLUCOSE] Biological tolerance-band filtering summary:
  tolerance=5pct: accepted 7820/8000 (97.75%)
  tolerance=2pct: accepted 7132/8000 (89.15%)
  tolerance=1pct: accepted 2050/8000 (25.62%)

Filtering complete. Log saved -> /kaggle/working/HPLC-DDPM-CNN/metrics/filtering_log_HPLC.txt


In [10]:
"""
==============================================================================
 CELL 10 — VALIDATION: PCA, t-SNE, CORRELATION HEATMAP, KS, WASSERSTEIN, JSD
==============================================================================
"""
def distrib_validation(real_flat, synth_flat, species_list, label, n_bins=20):
    rows = []
    for fi, col in enumerate(species_list):
        r = real_flat[:, fi]
        s = synth_flat[:, fi]
        ks_stat, ks_p = ks_2samp(r, s)
        w_dist = wasserstein_distance(r, s)
        
        lo, hi = min(r.min(), s.min()), max(r.max(), s.max())
        bins = np.linspace(lo, hi, n_bins + 1) if hi > lo else np.linspace(lo - 1, hi + 1, n_bins + 1)
        
        rh, _ = np.histogram(r, bins=bins, density=True)
        sh, _ = np.histogram(s, bins=bins, density=True)
        rh, sh = rh + 1e-12, sh + 1e-12
        rh, sh = rh / rh.sum(), sh / sh.sum()
        
        mh = 0.5 * (rh + sh)
        jsd = float(0.5 * entropy(rh, mh) + 0.5 * entropy(sh, mh))
        
        rows.append({
            'Subset': label, 'Species': col, 'KS_stat': ks_stat,
            'KS_p': ks_p, 'Wasserstein': w_dist, 'JSD': jsd
        })
    return pd.DataFrame(rows)

val_targets = ['OD'] + SPECIES
state = ddpm_state
real_arr = state['real_arr']  # Shape: (n_train_reps, 17, 6)
real_flat_valid = real_arr.reshape(-1, len(val_targets))

# ── 1. Statistical Distribution Validation Sweep ─────────────────────────────
val_dfs = []
for tag in FILTER_FRACTIONS:
    sub_df = filtered_subsets[tag]
    if sub_df['synthetic_id'].nunique() == 0:
        continue
    sub_flat = sub_df[val_targets].values.astype(float)
    vdf = distrib_validation(real_flat_valid, sub_flat, val_targets, tag)
    val_dfs.append(vdf)

if val_dfs:
    val_all = pd.concat(val_dfs, ignore_index=True)
    dist_path = os.path.join(MET_DIR, "distrib_validation_HPLC.xlsx")
    val_all.to_excel(dist_path, index=False)
    print("\n[HPLC-GLUCOSE] Distributional validation results:")
    print(val_all[['Subset', 'Species', 'KS_stat', 'Wasserstein', 'JSD']].to_string(index=False))

# ── 2. PCA & t-SNE Manifold Projections ──────────────────────────────────────
best_tag = max(FILTER_FRACTIONS, key=lambda tg: filtered_subsets[tg]['synthetic_id'].nunique())
best_sub = filtered_subsets[best_tag]

if best_sub['synthetic_id'].nunique() >= 2 and len(real_flat_valid) >= 2:
    synth_flat = best_sub[val_targets].values.astype(float)
    combined   = np.vstack([real_flat_valid, synth_flat])
    labels_arr = np.array(['Real'] * len(real_flat_valid) + ['Synthetic'] * len(synth_flat))

    pca_proj  = PCA(n_components=2, random_state=SEED).fit_transform(combined)
    perplex   = max(5, min(30, len(combined) // 4))
    tsne_proj = TSNE(n_components=2, random_state=SEED, perplexity=perplex,
                     n_iter=1000, init='pca').fit_transform(combined)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, proj, title in [(axes[0], pca_proj, "PCA"), (axes[1], tsne_proj, "t-SNE")]:
        ms, mr = labels_arr == 'Synthetic', labels_arr == 'Real'
        ax.scatter(proj[ms, 0], proj[ms, 1], c='#534AB7', s=12, alpha=0.3, label=f'Synthetic ({best_tag})')
        ax.scatter(proj[mr, 0], proj[mr, 1], c='#1D9E75', s=50, alpha=0.9,
                   edgecolors='white', linewidths=0.5, label='Real (Rep1+2)', zorder=10)
        ax.set_title(f"HPLC-GLUCOSE — {title}", fontweight='bold')
        ax.legend(fontsize=8, loc='best')

    fig.savefig(os.path.join(FIG_DIR, "Fig4_PCA_tSNE_HPLC.png"), **SAVEFIG_KW)
    plt.close(fig)

    # ── 3. Feature Correlation Heatmap Comparison ────────────────────────────
    real_corr  = pd.DataFrame(real_flat_valid, columns=val_targets).corr()
    synth_corr = pd.DataFrame(synth_flat, columns=val_targets).corr()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    for ax, cmat, title in [(axes[0], real_corr, "Real (Rep1+2)"), 
                            (axes[1], synth_corr, f"Synthetic ({best_tag})")]:
        sns.heatmap(cmat, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1,
                    square=True, linewidths=0.5, cbar_kws={'shrink': 0.75})
        ax.set_title(f"Correlation Matrix: {title}", fontweight='bold')

    fig.savefig(os.path.join(FIG_DIR, "Fig_CorrHeatmap_HPLC.png"), **SAVEFIG_KW)
    plt.close(fig)

    print("\nValidation figures successfully saved:")
    print("  -> Fig4_PCA_tSNE_HPLC.png")
    print("  -> Fig_CorrHeatmap_HPLC.png")


[HPLC-GLUCOSE] Distributional validation results:
Subset  Species  KS_stat  Wasserstein      JSD
  5pct       OD 0.039988     0.120418 0.022244
  5pct  Glucose 0.316880     0.075693 0.015436
  5pct   Xylose 0.055612     0.043033 0.017996
  5pct  Ethanol 0.076222     0.071856 0.019372
  5pct Glycerol 0.076899     0.008641 0.007428
  5pct  Acetate 0.105589     0.022627 0.022838
  2pct       OD 0.039911     0.112604 0.019319
  2pct  Glucose 0.314985     0.051688 0.014176
  2pct   Xylose 0.054980     0.037502 0.016091
  2pct  Ethanol 0.076020     0.062165 0.018704
  2pct Glycerol 0.075839     0.007420 0.005754
  2pct  Acetate 0.106776     0.021576 0.022407
  1pct       OD 0.039139     0.149099 0.012410
  1pct  Glucose 0.300172     0.061221 0.009292
  1pct   Xylose 0.109641     0.062176 0.021081
  1pct  Ethanol 0.136987     0.130929 0.044424
  1pct Glycerol 0.118451     0.012350 0.023816
  1pct  Acetate 0.113888     0.038427 0.059600

Validation figures successfully saved:
  -> Fig4_PCA_tS

In [11]:
"""
==============================================================================
 CELL 11 — CNN WINDOW CONSTRUCTION & FEATURE SCALING FOR 1D-CNN (ROBUST)
 Fits feature scaler and target rate scaler on Real Training (Replica-1 + Replica-2).
 Constructs 3D sliding sequence windows for real and synthetic datasets.
==============================================================================
"""
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# ── 0. Define Target Species and Column Mappings ───────────────────────────
if 'ALL_SPECIES' in globals():
    species_list = ALL_SPECIES
elif 'DDPM_TARGETS' in globals():
    species_list = DDPM_TARGETS
else:
    species_list = ['OD', 'Glucose', 'Xylose', 'Ethanol', 'Glycerol', 'Acetate']

WINDOW_SIZE = globals().get('WINDOW_SIZE', 3)
TIME_COL    = globals().get('TIME_COL', 'Hour')

rep1_cols = processed_replicates['Replica-1'].columns.tolist()

# Match input feature columns (_smooth or raw)
input_cols = []
for sp in species_list:
    if f"{sp}_smooth" in rep1_cols:
        input_cols.append(f"{sp}_smooth")
    elif sp in rep1_cols:
        input_cols.append(sp)

INPUT_FEATURES = [TIME_COL] + input_cols

# Match target rate columns (_rate or d/dt)
TARGET_RATES = []
for sp in species_list:
    if f"{sp}_rate" in rep1_cols:
        TARGET_RATES.append(f"{sp}_rate")
    elif f"d{sp}/dt" in rep1_cols:
        TARGET_RATES.append(f"d{sp}/dt")

n_in_features  = len(INPUT_FEATURES)
n_out_features = len(TARGET_RATES)

print(f"[HPLC-GLUCOSE] 1D-CNN Features ({n_in_features}): {INPUT_FEATURES}")
print(f"[HPLC-GLUCOSE] Target Rates ({n_out_features}): {TARGET_RATES}")

# ── 1. Fit Scalers strictly on Real Training Data (Replica-1 + Replica-2) ────
train_real_df = pd.concat([processed_replicates['Replica-1'], 
                           processed_replicates['Replica-2']], ignore_index=True)

feature_scaler = StandardScaler()
feature_scaler.fit(train_real_df[INPUT_FEATURES].values)

# Scaler for target rates to ensure balanced multi-target MSE loss
rate_scaler = MinMaxScaler(feature_range=(-1, 1))
rate_scaler.fit(train_real_df[TARGET_RATES].values)

# ── 2. Robust Sliding Window Generator Function ───────────────────────────────
def make_sliding_windows(df: pd.DataFrame, group_col=None, window_size=3):
    X_list, Y_list = [], []
    
    if group_col and group_col in df.columns:
        grouped = df.groupby(group_col, sort=False)
    else:
        grouped = [(None, df)]
        
    for _, grp in grouped:
        grp = grp.sort_values(TIME_COL).reset_index(drop=True).copy()
        if len(grp) < window_size:
            continue
            
        # Ensure smooth feature names exist
        missing_inputs = [c for c in INPUT_FEATURES if c not in grp.columns]
        if missing_inputs:
            rename_dict = {sp: f"{sp}_smooth" for sp in species_list if sp in grp.columns}
            grp = grp.rename(columns=rename_dict)
            
        x_scaled = feature_scaler.transform(grp[INPUT_FEATURES].values)
        
        # Compute missing rates dynamically if needed
        missing_targets = [c for c in TARGET_RATES if c not in grp.columns]
        if missing_targets:
            for sp, r_col in zip(species_list, TARGET_RATES):
                conc_col = f"{sp}_smooth" if f"{sp}_smooth" in grp.columns else sp
                if conc_col in grp.columns:
                    grp[r_col] = np.gradient(grp[conc_col].values, grp[TIME_COL].values)
                else:
                    grp[r_col] = 0.0
                    
        y_vals = grp[TARGET_RATES].values
        
        for i in range(len(grp) - window_size + 1):
            X_list.append(x_scaled[i : i + window_size])
            # Target rate taken at final time-step of window
            Y_list.append(y_vals[i + window_size - 1])
            
    if not X_list:
        return np.empty((0, window_size, n_in_features), dtype=np.float32), np.empty((0, n_out_features), dtype=np.float32)
        
    return np.array(X_list, dtype=np.float32), np.array(Y_list, dtype=np.float32)

# ── 3. Construct Windowed Datasets for Real Replicates ───────────────────────
X_train_real, Y_train_real = make_sliding_windows(train_real_df, window_size=WINDOW_SIZE)
X_test_real, Y_test_real   = make_sliding_windows(processed_replicates['Replica-3'], window_size=WINDOW_SIZE)

print(f"\n[HPLC-GLUCOSE] Real Window Datasets Generated:")
print(f"  Real Train (Rep1+2) : X={X_train_real.shape}, Y={Y_train_real.shape}")
print(f"  Real Test  (Rep3)   : X={X_test_real.shape}, Y={Y_test_real.shape}")

# ── 4. Construct Windowed Datasets for Synthetic Subsets ────────────────────
cnn_datasets = {
    'Real-Only': (X_train_real, Y_train_real),
    'Baseline-Cand': make_sliding_windows(synthetic_candidates, group_col='synthetic_id', window_size=WINDOW_SIZE)
}

for tag in FILTER_FRACTIONS:
    if tag in filtered_subsets:
        sub_df = filtered_subsets[tag]
        if len(sub_df) > 0:
            X_sub, Y_sub = make_sliding_windows(sub_df, group_col='synthetic_id', window_size=WINDOW_SIZE)
            if len(X_sub) > 0:
                X_aug = np.concatenate([X_train_real, X_sub], axis=0)
                Y_aug = np.concatenate([Y_train_real, Y_sub], axis=0)
                cnn_datasets[f'Augmented-{tag}'] = (X_aug, Y_aug)
                print(f"  Augmented-{tag:<4s}     : X={X_aug.shape}, Y={Y_aug.shape} "
                      f"(Real: {len(X_train_real)} + Synthetic: {len(X_sub)})")

print("\n1D-CNN window datasets constructed and feature scaling complete.")

[HPLC-GLUCOSE] 1D-CNN Features (7): ['Hour', 'OD_smooth', 'Glucose_smooth', 'Xylose_smooth', 'Ethanol_smooth', 'Glycerol_smooth', 'Acetate_smooth']
[HPLC-GLUCOSE] Target Rates (6): ['OD_rate', 'Glucose_rate', 'Xylose_rate', 'Ethanol_rate', 'Glycerol_rate', 'Acetate_rate']

[HPLC-GLUCOSE] Real Window Datasets Generated:
  Real Train (Rep1+2) : X=(32, 3, 7), Y=(32, 6)
  Real Test  (Rep3)   : X=(15, 3, 7), Y=(15, 6)
  Augmented-5pct     : X=(117332, 3, 7), Y=(117332, 6) (Real: 32 + Synthetic: 117300)
  Augmented-2pct     : X=(107012, 3, 7), Y=(107012, 6) (Real: 32 + Synthetic: 106980)
  Augmented-1pct     : X=(30782, 3, 7), Y=(30782, 6) (Real: 32 + Synthetic: 30750)

1D-CNN window datasets constructed and feature scaling complete.


In [12]:
"""
==============================================================================
 CELL 12 — 1D-CNN MODEL TRAINING, EVALUATION & KINETIC RATE COMPARISON (FIXED)
 Predicts kinetic rates (dY/dt) for biomass (OD) and all 5 metabolites.
 Incorporates target scaling, sample weighting, and fine-tuning.
==============================================================================
"""
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── 1. Define 1D-CNN Multi-Target Regressor ─────────────────────────────────
class CNN1DRegressor(nn.Module):
    def __init__(self, in_channels=7, out_channels=6, conv_filters=(32, 64), fc_dims=(64, 32)):
        super().__init__()
        layers = []
        c_in = in_channels
        for c_out in conv_filters:
            layers.extend([
                nn.Conv1d(c_in, c_out, kernel_size=2, padding=1),
                nn.BatchNorm1d(c_out),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            c_in = c_out
        self.conv_net = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        fc_layers = []
        f_in = conv_filters[-1]
        for f_out in fc_dims:
            fc_layers.extend([
                nn.Linear(f_in, f_out),
                nn.ReLU(),
                nn.Dropout(0.1)
            ])
            f_in = f_out
        fc_layers.append(nn.Linear(f_in, out_channels))
        self.fc_net = nn.Sequential(*fc_layers)
        
    def forward(self, x):
        # Transpose input shape from (Batch, SeqLen, Channels) to (Batch, Channels, SeqLen)
        x = x.permute(0, 2, 1)
        feat = self.conv_net(x)
        feat = self.gap(feat).squeeze(-1)
        return self.fc_net(feat)

# ── 2. Training Helper Function with Target Scaling & Sample Weighting ───────
def train_eval_cnn(X_tr, Y_tr, X_te, Y_te, epochs=300, lr=1e-3, batch_size=256, n_real_samples=32):
    model = CNN1DRegressor(in_channels=n_in_features, out_channels=n_out_features).to(DEVICE)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    
    # Scale targets to [-1, 1] using fitted rate_scaler to balance loss across channels
    Y_tr_scaled = rate_scaler.transform(Y_tr)
    
    # Sample weighting: Assign 10x higher weight to real experimental training samples
    weights = np.ones(len(X_tr), dtype=np.float32)
    if len(X_tr) > n_real_samples:
        weights[:n_real_samples] = 10.0
        
    X_tr_t = torch.from_numpy(X_tr).float()
    Y_tr_t = torch.from_numpy(Y_tr_scaled).float()
    W_tr_t = torch.from_numpy(weights).float()
    
    X_te_t = torch.from_numpy(X_te).float()
    
    dataset = TensorDataset(X_tr_t, Y_tr_t, W_tr_t)
    loader  = DataLoader(dataset, batch_size=min(batch_size, len(X_tr)), shuffle=True, drop_last=False)
    
    for ep in range(epochs):
        model.train()
        for bx, by, bw in loader:
            bx, by, bw = bx.to(DEVICE), by.to(DEVICE), bw.to(DEVICE)
            pred = model(bx)
            # Weighted MSE loss
            loss = (bw.unsqueeze(1) * (pred - by)**2).mean()
            
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
            
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_te_t.to(DEVICE)).cpu().numpy()
        
    # Inverse transform predictions back to physical rate units (g/L/h or OD/h)
    preds = rate_scaler.inverse_transform(preds_scaled)
    return model, preds

# ── 3. Benchmark Evaluation across Dataset Variants ──────────────────────────
results_rows = []
species_rows = []
model_predictions = {}

print("[HPLC-GLUCOSE] Training and evaluating 1D-CNN predictive models ...")

n_real_windows = len(X_train_real)

for name, (X_tr, Y_tr) in cnn_datasets.items():
    model, preds = train_eval_cnn(
        X_tr, Y_tr, X_test_real, Y_test_real, 
        epochs=350, lr=1e-3, batch_size=256, n_real_samples=n_real_windows
    )
    model_predictions[name] = preds
    
    # Calculate overall metrics across all channels
    r2_overall   = r2_score(Y_test_real, preds)
    rmse_overall = np.sqrt(mean_squared_error(Y_test_real, preds))
    mae_overall  = mean_absolute_error(Y_test_real, preds)
    
    results_rows.append({
        'Dataset': name,
        'N_Train_Windows': len(X_tr),
        'R2_Overall': r2_overall,
        'RMSE_Overall': rmse_overall,
        'MAE_Overall': mae_overall
    })
    
    # Calculate per-species metrics
    for idx, target in enumerate(TARGET_RATES):
        sp_name = target.replace('_rate', '').replace('d', '').replace('/dt', '')
        r2_sp   = r2_score(Y_test_real[:, idx], preds[:, idx])
        rmse_sp = np.sqrt(mean_squared_error(Y_test_real[:, idx], preds[:, idx]))
        mae_sp  = mean_absolute_error(Y_test_real[:, idx], preds[:, idx])
        
        species_rows.append({
            'Dataset': name,
            'Species': sp_name,
            'R2': r2_sp,
            'RMSE': rmse_sp,
            'MAE': mae_sp
        })

# Export evaluation tables
df_results = pd.DataFrame(results_rows)
df_species = pd.DataFrame(species_rows)

excel_met_path = os.path.join(MET_DIR, "cnn_evaluation_HPLC.xlsx")
with pd.ExcelWriter(excel_met_path) as writer:
    df_results.to_excel(writer, sheet_name="Overall_Metrics", index=False)
    df_species.to_excel(writer, sheet_name="Per_Species_Metrics", index=False)

print("\n=== [HPLC-GLUCOSE] 1D-CNN Overall Test Performance (Replica-3) ===")
print(df_results.to_string(index=False))

# ── 4. Save Held-Out Kinetic Rate Predictions Plot ─────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
axes = axes.flatten()

# Time steps corresponding to window targets
test_time = processed_replicates['Replica-3'][TIME_COL].values[WINDOW_SIZE - 1:]

palette = {
    'Real-Only': '#D95F02', 
    'Baseline-Cand': '#7570B3',
    'Augmented-5pct': '#E7298A', 
    'Augmented-2pct': '#66A61E', 
    'Augmented-1pct': '#1B9E77'
}

for idx, target in enumerate(TARGET_RATES):
    ax = axes[idx]
    sp_name = target.replace('_rate', '').replace('d', '').replace('/dt', '')
    
    # Plot ground truth rates from Replica-3
    ax.plot(test_time, Y_test_real[:, idx], color='black', lw=2.5, marker='o',
            label='True (Replica-3)', zorder=10)
    
    # Plot model predictions
    for d_name, preds in model_predictions.items():
        ax.plot(test_time, preds[:, idx], lw=1.5, ls='--', 
                color=palette.get(d_name, '#333333'), label=d_name)
        
    ax.set_title(f"Rate d({sp_name})/dt", fontweight='bold')
    ax.set_xlabel("Time (Hour)")
    ax.set_ylabel("Rate (g/L/h)")
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.2)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Fig5_KineticRates_HPLC.png"), **SAVEFIG_KW)
plt.close(fig)

print(f"\nEvaluation complete. Metrics -> {excel_met_path}")
print("Kinetic rate prediction plot saved -> Fig5_KineticRates_HPLC.png")

[HPLC-GLUCOSE] Training and evaluating 1D-CNN predictive models ...

=== [HPLC-GLUCOSE] 1D-CNN Overall Test Performance (Replica-3) ===
       Dataset  N_Train_Windows  R2_Overall  RMSE_Overall  MAE_Overall
     Real-Only               32    0.688850      0.219016     0.106579
 Baseline-Cand           120000    0.616807      0.174298     0.092857
Augmented-5pct           117332    0.708639      0.163423     0.078492
Augmented-2pct           107012    0.493521      0.353003     0.129320
Augmented-1pct            30782    0.697937      0.160701     0.078138

Evaluation complete. Metrics -> /kaggle/working/HPLC-DDPM-CNN/metrics/cnn_evaluation_HPLC.xlsx
Kinetic rate prediction plot saved -> Fig5_KineticRates_HPLC.png


In [13]:
"""
==============================================================================
 CELL 13 — SUMMARY REPORT & ARTIFACT VERIFICATION (FIXED)
 Consolidates pipeline metrics, prints final performance summary, and 
22232223356 verifies all exported figures, datasets, and logs.
====================================================234\==========================
"""
import os
import pandas as pd

# ── 0. Safe Fallback Resolvers ───────────────────────────────────────────────
BASE_DIR = globals().get('BASE', "/kaggle/working/HPLC-DDPM-CNN")
FIG_DIR  = globals().get('FIG_DIR', os.path.join(BASE_DIR, "figures"))
SYN_DIR  = globals().get('SYN_DIR', os.path.join(BASE_DIR, "synthetic_data"))
MET_DIR  = globals().get('MET_DIR', os.path.join(BASE_DIR, "metrics"))

excel_met_path = globals().get(
    'excel_met_path', 
    os.path.join(MET_DIR, "cnn_evaluation_HPLC.xlsx")
)

print("=" * 80)
print("              HPLC-GLUCOSE PIPELINE EXECUTION COMPLETE               ")
print("=" * 80)

# ── 1. Display Consolidated Overall & Per-Species Performance Tables ──────────
if os.path.exists(excel_met_path):
    print("\n[1D-CNN Overall Model Evaluation Summary on Held-out Replica-3]")
    print("-" * 80)
    summary_df = pd.read_excel(excel_met_path, sheet_name="Overall_Metrics")
    print(summary_df.to_string(index=False))
    print("-" * 80)

    # Identify best performing variant by R2 score
    best_variant = summary_df.loc[summary_df['R2_Overall'].idxmax()]
    print(f"\nTop Performing Pipeline : {best_variant['Dataset']}")
    print(f"  Overall R² Score      : {best_variant['R2_Overall']:.4f}")
    print(f"  Overall RMSE          : {best_variant['RMSE_Overall']:.4f}")
    print(f"  Overall MAE           : {best_variant['MAE_Overall']:.4f}")

    # Display per-species metrics summary if available
    try:
        species_df = pd.read_excel(excel_met_path, sheet_name="Per_Species_Metrics")
        print("\n[Per-Species Breakdown for Top Performing Pipeline]")
        print("-" * 80)
        best_species = species_df[species_df['Dataset'] == best_variant['Dataset']]
        print(best_species[['Species', 'R2', 'RMSE', 'MAE']].to_string(index=False))
        print("-" * 80)
    except Exception as e:
        pass
else:
    print(f"\n⚠️ Metrics summary file not found at: {excel_met_path}")

# ── 2. Directory Artifact Verification ───────────────────────────────────────
print("\n[Exported Output Artifacts Directory Check]")
print("-" * 80)

dir_checks = [
    ("Figures", FIG_DIR),
    ("Synthetic Data", SYN_DIR),
    ("Metrics & Logs", MET_DIR)
]

for label, path in dir_checks:
    print(f"\n{label} Directory ({path}):")
    if os.path.exists(path):
        files = os.listdir(path)
        if files:
            for f in sorted(files):
                f_path = os.path.join(path, f)
                f_size = os.path.getsize(f_path) / 1024.0  # size in KB
                print(f"   ├── {f:<35s} ({f_size:.1f} KB)")
        else:
            print("   └── Directory is empty.")
    else:
        print("   └── Directory not found.")

print("\n" + "=" * 80)
print("Pipeline complete. All synthetic trajectories, figures, and model evaluations")
print(f"are successfully exported to {BASE_DIR}/")
print("=" * 80)

              HPLC-GLUCOSE PIPELINE EXECUTION COMPLETE               

[1D-CNN Overall Model Evaluation Summary on Held-out Replica-3]
--------------------------------------------------------------------------------
       Dataset  N_Train_Windows  R2_Overall  RMSE_Overall  MAE_Overall
     Real-Only               32    0.688850      0.219016     0.106579
 Baseline-Cand           120000    0.616807      0.174298     0.092857
Augmented-5pct           117332    0.708639      0.163423     0.078492
Augmented-2pct           107012    0.493521      0.353003     0.129320
Augmented-1pct            30782    0.697937      0.160701     0.078138
--------------------------------------------------------------------------------

Top Performing Pipeline : Augmented-5pct
  Overall R² Score      : 0.7086
  Overall RMSE          : 0.1634
  Overall MAE           : 0.0785

[Per-Species Breakdown for Top Performing Pipeline]
--------------------------------------------------------------------------------
 S

In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error

# ── 1. Set Journal Publication Styling Parameters ───────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'axes.linewidth': 1.1,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
    'figure.autolayout': False
})

# ── 2. Data Alignment ────────────────────────────────────────────────────────
metabolites = ['OD'] + SPECIES
n_met = len(metabolites)

real_mean_df = pd.DataFrame({TIME_COL: processed_replicates['Replica-1'][TIME_COL]})
for sp in metabolites:
    real_mean_df[sp] = np.mean([
        processed_replicates['Replica-1'][sp].values,
        processed_replicates['Replica-2'][sp].values,
        processed_replicates['Replica-3'][sp].values
    ], axis=0)

syn_mean_df = synthetic_candidates.groupby(TIME_COL)[metabolites].mean().reset_index()

# ── 3. Plot Publication Parity Grid ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(10.5, 7.2), dpi=300)
axes = axes.flatten()

markers = {'Replica-1': 'o', 'Replica-2': 's', 'Replica-3': '^', 'Synthetic': 'D'}
colors  = {'Replica-1': '#1F77B4', 'Replica-2': '#FF7F0E', 'Replica-3': '#2CA02C', 'Synthetic': '#E377C2'}

for i, sp in enumerate(metabolites):
    ax = axes[i]
    exp_ref = real_mean_df[sp].values   # Ground Truth (X)
    syn_pred = syn_mean_df[sp].values   # Model Synthetic (Y)
    
    # Calculate Quantitative Metrics
    r2_syn = r2_score(exp_ref, syn_pred)
    rmse_syn = np.sqrt(mean_squared_error(exp_ref, syn_pred))
    
    # Plot experimental replicates
    for rep in REPLICATES:
        rep_vals = processed_replicates[rep][sp].values
        ax.scatter(exp_ref, rep_vals, color=colors[rep], marker=markers[rep], 
                   s=24, alpha=0.75, edgecolors='none', label=rep)
        
    # Plot synthetic predictions
    ax.scatter(exp_ref, syn_pred, color=colors['Synthetic'], marker=markers['Synthetic'], 
               s=28, alpha=0.85, edgecolors='none', label='Synthetic (Mean)')

    # Diagonal Parity Line
    all_vals = np.concatenate([exp_ref, syn_pred] + [processed_replicates[r][sp].values for r in REPLICATES])
    vmin, vmax = np.min(all_vals), np.max(all_vals)
    padding = 0.05 * (vmax - vmin) if vmax != vmin else 0.1
    lims = [vmin - padding, vmax + padding]
    
    ax.plot(lims, lims, color='black', linestyle='--', linewidth=1.0, alpha=0.8)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal', adjustable='box')
    
    unit = "OD600" if sp == 'OD' else "g/L"
    
    # Subplot Title & Inset Metrics (Top-Left, matching paper style)
    ax.set_title(f"{sp} ({unit})", fontweight='bold', pad=6)
    
    stats_text = f"$R^2$: {r2_syn:.3f}\nRMSE: {rmse_syn:.3f}"
    ax.text(0.05, 0.92, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='left')
    
    # Paper-style Spines & Inward Ticks
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)
    
    ax.tick_params(direction='in', top=True, right=True, length=4, width=1.0)
    ax.grid(False)  # Clean canvas behind points

# ── 4. Global Axis Labels & Legend Alignment ────────────────────────────────
fig.text(0.5, 0.03, 'Experimental Observed', ha='center', va='center', fontweight='bold', fontsize=13)
fig.text(0.02, 0.5, 'Predicted / Synthetic', ha='center', va='center', rotation='vertical', fontweight='bold', fontsize=13)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.04),
    ncol=4,
    frameon=True,
    facecolor='white',
    edgecolor='#cccccc',
    fontsize=11
)

plt.tight_layout(rect=[0.04, 0.05, 1, 1])

# ── 5. Output Export ─────────────────────────────────────────────────────────
save_path_png = os.path.join(FIG_DIR, "Fig_Parity_Plots_Metabolites.png")
save_path_pdf = os.path.join(FIG_DIR, "Fig_Parity_Plots_Metabolites.pdf")
save_path_svg = os.path.join(FIG_DIR, "Fig_Parity_Plots_Metabolites.svg")

fig.savefig(save_path_png, dpi=600, bbox_inches='tight', transparent=False)
fig.savefig(save_path_pdf, format='pdf', bbox_inches='tight')
fig.savefig(save_path_svg, format='svg', bbox_inches='tight')

plt.close(fig)
print(f"Publication parity figure re-exported successfully:\n - {save_path_png}\n - {save_path_pdf}\n - {save_path_svg}")

Publication parity figure re-exported successfully:
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_Parity_Plots_Metabolites.png
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_Parity_Plots_Metabolites.pdf
 - /kaggle/working/HPLC-DDPM-CNN/figures/Fig_Parity_Plots_Metabolites.svg
